In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Configuration
NB_LIGNES = 1_000_000  # 1 Million !
TAUX_FRAUDE = 0.05     # 5% de fraude (plus réaliste)

print(f"🚀 Génération de {NB_LIGNES} lignes en cours... (Patience !)")

# 1. Génération des ID et Dates
data = {
    'TransactionID': np.arange(NB_LIGNES),
    'DateTransaction': [datetime(2025, 1, 1) + timedelta(days=random.randint(0, 365), hours=random.randint(0, 23)) for _ in range(NB_LIGNES)]
}

# 2. Variables Catégorielles (Listes de choix)
villes = ['Port-au-Prince', 'Carrefour', 'Delmas', 'Pétion-Ville', 'Cap-Haïtien', 'Gonaïves', 'Jacmel']
devices = ['Mobile', 'Web', 'Tablet', 'Inconnu']
marchands = ['Supermarché', 'Station-service', 'Restaurant', 'Pharmacie', 'En ligne', 'Hôtel']

# On utilise numpy choice pour aller très vite
data['Ville'] = np.random.choice(villes, NB_LIGNES)
data['Device'] = np.random.choice(devices, NB_LIGNES, p=[0.6, 0.3, 0.05, 0.05])
data['TypeMarchand'] = np.random.choice(marchands, NB_LIGNES)

# 3. Variables Numériques (Distributions normales)
data['Age'] = np.random.randint(18, 90, NB_LIGNES)
data['Anciennete_Jours'] = np.random.randint(1, 4000, NB_LIGNES)
data['Revenu_Mensuel'] = np.random.normal(50000, 15000, NB_LIGNES).clip(min=5000) # Moyenne 50k, écart 15k

# 4. LA CIBLE : FRAUDE (0 ou 1)
# On génère 5% de fraude aléatoirement
data['Fraude'] = np.random.choice([0, 1], NB_LIGNES, p=[1-TAUX_FRAUDE, TAUX_FRAUDE])

# --- CRÉATION DE MOTIFS (PATTERNS) POUR QUE LE MODÈLE APPRENNE ---
# Si c'est une fraude, on change les valeurs pour qu'elles soient "suspectes"
# Sinon le modèle ne pourra rien prédire du tout.

df = pd.DataFrame(data)

# Motif 1 : Les fraudes ont souvent des montants plus élevés ou très petits
# On génère des montants normaux
df['Montant'] = np.random.exponential(1000, NB_LIGNES) 

# On modifie les montants pour les lignes de fraude (Fraude = 1)
# Les fraudeurs dépensent soit beaucoup (x10), soit testent des petites sommes
mask_fraude = df['Fraude'] == 1
df.loc[mask_fraude, 'Montant'] = df.loc[mask_fraude, 'Montant'] * np.random.choice([0.1, 10, 20], sum(mask_fraude))

# Motif 2 : Les fraudes arrivent souvent la nuit (entre 1h et 4h du matin)
# On extrait l'heure
df['Heure'] = df['DateTransaction'].apply(lambda x: x.hour)
# Si fraude, on force souvent une heure tardive
df.loc[mask_fraude, 'Heure'] = np.random.choice([1, 2, 3, 4, 23], sum(mask_fraude))

# Motif 3 : Les fraudes viennent souvent de "Device Inconnu"
df.loc[mask_fraude, 'Device'] = np.random.choice(['Inconnu', 'Web'], sum(mask_fraude), p=[0.7, 0.3])

# 5. Nettoyage final et Export
print("💾 Sauvegarde dans 'dataset_million_fraude.csv'...")
df.to_csv('dataset_million_fraude.csv', index=False)
df.to_excel('dataset_million_fraude.xlsx', index=False)
print("✅ Terminé ! Tu as maintenant 1 million de lignes.")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix

# ==============================================================================
# TEST DE PERFORMANCE : DATASET 1 MILLION DE LIGNES
# ==============================================================================

print("1. Chargement du monstre (1 Million de lignes)...")
# On charge le fichier que tu viens de creer
df_big = pd.read_csv('dataset_million_fraude.csv')
print(f"-> Charge ! Taille : {df_big.shape}")

# 2. Préparation (Adaptée aux colonnes de ce fichier synthétique)
# On exclut les colonnes inutiles pour le modèle
X = df_big.drop(columns=['Fraude', 'TransactionID', 'DateTransaction'])
y = df_big['Fraude']

# Définition des colonnes (basé sur le script générateur)
numeric_features = ['Montant', 'Age', 'Anciennete_Jours', 'Revenu_Mensuel', 'Heure']
categorical_features = ['Ville', 'Device', 'TypeMarchand']

# 3. Création d'un Pipeline "Express"
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# LE MODÈLE PUISSANT : Random Forest
# n_jobs=-1 permet d'utiliser tous les processeurs de ton ordi pour aller plus vite
model_big = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=50, max_depth=15, n_jobs=-1, random_state=42))
])

# 4. Séparation Train/Test
print("2. Separation Train/Test...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Entraînement
print("3. Entrainement en cours...")
model_big.fit(X_train, y_train)

# 6. Résultats
print("4. Evaluation...")
y_pred = model_big.predict(X_test)

print("\n--- RESULTATS ---")
print(classification_report(y_test, y_pred))

# Petit bonus : Matrice de confusion simplifiée
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print(f"\nFraudes detectées (Vrais Positifs) : {tp}")
print(f"Fraudes ratées (Faux Négatifs)     : {fn}")
print(f"Clients innocents bloqués (Faux Positifs) : {fp}")